# ✂️ Step 1: 회의록 TSV 배치 분할 및 전역 row_id 부여 (`split_council_tsv`)
대규모 회의록 TSV 파일을 로컬 LLM이 처리하기 적합한 소형 청크 단위(행 수/문자 수 기준)로 분할합니다.

### 📌 왜 분할이 필요한가?
- **LLM 컨텍스트 윈도우 및 시간 초과 방지**: 수만 행의 발언을 한 번에 처리할 경우 메모리 오버헤드와 모델 타임아웃 발생
- **전역 `row_id` 부여**: 각 발언에 분할 전 원본 기준 1부터 증가하는 고유 `row_id`를 영구 부여하여, 별도로 메타데이터를 추출해도 원본 데이터와 100% 무결하게 조인 가능
- **멀티스레드 병렬 및 실패 격리**: 특정 배치가 실패하더라도 해당 배치만 재시도 가능

In [ ]:
import csv
import glob
import os
from pathlib import Path
import pandas as pd

# ---------------------------------------------------------------------------
# 1. 환경 설정 및 경로 지정
# ---------------------------------------------------------------------------
COUNCIL_NAME = "강원특별자치도의회"
TARGET_YEAR = "2024"

# 입력 TSV 후보 자동 탐색
candidate_inputs = [
    Path(f"../../data/to_text2/{TARGET_YEAR}_광역 회의록/cleaned_{TARGET_YEAR}_{COUNCIL_NAME}.txt"),
    Path(f"../../text/cleaned_{TARGET_YEAR}_{COUNCIL_NAME}.txt"),
    Path(f"./data/to_text2/{TARGET_YEAR}_광역 회의록/cleaned_{TARGET_YEAR}_{COUNCIL_NAME}.txt"),
    Path(f"./text/cleaned_{TARGET_YEAR}_{COUNCIL_NAME}.txt"),
]
INPUT_FILE = next((p for p in candidate_inputs if p.exists()), candidate_inputs[0])

# 분할 파일 저장 폴더
OUTPUT_DIR = Path(f"./data/to_text/{TARGET_YEAR}_광역 회의록/split/{COUNCIL_NAME}")
ROWS_PER_FILE = 20      # 파일당 최대 행 수
MAX_CHARS = 10000       # 파일당 최대 문자 수 (긴 발언 포함 시 행 수 유동 조정)

print(f"입력 파일: {INPUT_FILE.resolve() if INPUT_FILE.exists() else '존재 확인 필요: ' + str(INPUT_FILE)}")
print(f"분할 폴더: {OUTPUT_DIR.resolve()}")

## 2. 스키마 검증 및 분할 배치 작성기

In [ ]:
ENGLISH_COLUMNS = ["date", "assembly", "agenda_title", "speaker_role", "speaker_name", "speaker_type", "utterance"]
KOREAN_COLUMNS = ["MTG_DE", "RASMBLY_NM", "MTGNM", "MINTS_NM", "agenda_title", "speaker_role", "speaker_name", "speaker_type", "utterance"]

def parse_header(reader):
    header = next(reader)
    header = [c.strip().lstrip("\ufeff") for c in header]
    if header == ENGLISH_COLUMNS or header == KOREAN_COLUMNS:
        return header
    if "utterance" in header and ("speaker_name" in header or "speaker" in header):
        return header
    raise ValueError(f"예상치 못한 헤더: {header}")

def split_tsv(input_path: Path, output_dir: Path, rows_per_file: int = 20, max_chars: int = 10000):
    if not input_path.is_file():
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {input_path}")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    with input_path.open("r", encoding="utf-8-sig", newline="") as src:
        reader = csv.reader(src, delimiter="\t")
        header = parse_header(reader)
        
        batch = []
        batch_chars = 0
        batch_no = 0
        total_rows = 0
        
        def flush():
            nonlocal batch, batch_chars, batch_no
            if not batch:
                return
            batch_no += 1
            out_p = output_dir / f"part_{batch_no:05d}.tsv"
            with out_p.open("w", encoding="utf-8-sig", newline="") as out_f:
                w = csv.writer(out_f, delimiter="\t", lineterminator="\n")
                w.writerow(["row_id", *header])
                for r_id, fields in batch:
                    w.writerow([r_id, *fields])
            batch = []
            batch_chars = 0

        row_id = 0
        for line_no, row in enumerate(reader, start=2):
            if not row or (len(row) == 1 and not row[0].strip()):
                continue
            if len(row) < len(header):
                continue
            row_id += 1
            r_chars = sum(len(f) for f in row)
            if batch and (len(batch) >= rows_per_file or batch_chars + r_chars > max_chars):
                flush()
            batch.append((row_id, row[:len(header)]))
            batch_chars += r_chars
            total_rows += 1
            
        flush()
    print(f"\n✓ 분할 완료: 총 {total_rows:,}개 행 -> {batch_no:,}개 파일 (폴더: {output_dir.name})")
    return total_rows, batch_no

## 3. 분할 실행 및 결과 검증

In [ ]:
if INPUT_FILE.exists():
    total_rows, total_batches = split_tsv(
        input_path=INPUT_FILE,
        output_dir=OUTPUT_DIR,
        rows_per_file=ROWS_PER_FILE,
        max_chars=MAX_CHARS
    )
else:
    print(f"⚠️ 테스트를 위해 입력 파일이 필요합니다. 임시 TSV를 생성하여 시연합니다.")
    dummy_input = Path("./sample_test.tsv")
    dummy_input.write_text(
        "date\tassembly\tagenda_title\tspeaker_role\tspeaker_name\tspeaker_type\tutterance\n" +
        "20240123\t강원특별자치도의회\t농지 조례안\t위원장\t김철수\t의회\t성원이 되었으므로 회의를 속개하겠습니다.\n" +
        "20240123\t강원특별자치도의회\t농지 조례안\t의원\t이영희\t의회\t농지제도 규제 완화를 통해 청년농 유입을 촉진해야 합니다.\n",
        encoding="utf-8"
    )
    split_tsv(dummy_input, OUTPUT_DIR, rows_per_file=1)

# 분할된 파일 확인
part_files = sorted(OUTPUT_DIR.glob("part_*.tsv"))
print(f"생성된 분할 파일 목록 (처음 5개):")
for p in part_files[:5]:
    df_part = pd.read_csv(p, sep="\t")
    print(f"  - {p.name}: {len(df_part)}행 (row_id: {df_part['row_id'].iloc[0]} ~ {df_part['row_id'].iloc[-1]})")